In [1]:
import pandas as pd
import json

df_train = pd.read_csv('data/codex/train.txt', sep='\t', header=None, names=['Head','Relation','Tail'])
df_test = pd.read_csv('data/codex/test.txt', sep='\t', header=None, names=['Head','Relation','Tail'])
df_valid = pd.read_csv('data/codex/valid.txt', sep='\t', header=None, names=['Head','Relation','Tail'])

with open('data/codex/entities.json', 'r') as file:
    entities = json.load(file)

with open('data/codex/relations.json', 'r') as file:
    relations = json.load(file)

def id2name_df(id_df:pd.DataFrame, entities_dict:dict, relations_dict:dict)-> pd.DataFrame:
    name_dict = {}
    for id, row in id_df.iterrows():
        # get id
        head_id = row['Head']
        relation_id = row['Relation']
        tail_id = row['Tail']
        # get label out of id
        head = entities_dict[head_id]['label']
        relation = relations_dict[relation_id]['label']
        tail = entities_dict[tail_id]['label']
        name_dict[id] = [head,relation,tail]
    name_df = pd.DataFrame.from_dict(name_dict,orient='index',columns=['Head','Relation','Tail'])
    return name_df

df_train_name = id2name_df(df_train, entities, relations)
df_test_name = id2name_df(df_test, entities, relations)
df_valid_name = id2name_df(df_valid, entities, relations)

df_name = pd.concat([df_train_name, df_test_name, df_valid_name]).reset_index(drop=True)

/var/folders/_3/wtwzgv1d3rlfz233qkf36kg00000gp/T/ipykernel_24913/403170735.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [8]:
from cand_gen.embedding import train_model
from cand_gen import triple_gen
import cand_gen.embedding
import importlib
from cand_gen.embedding.get_emb_transe import get_list_dist
importlib.reload(cand_gen.embedding)

<module 'cand_gen.embedding' (namespace) from ['/Users/fieng/Project/KGC-llm/cand_gen/embedding', '/Users/fieng/Project/KGC-llm/cand_gen/embedding']>

In [ ]:
df_name_sample = df_name.sample(int(0.8*len(df_name)))
train_df = train_model.create_dataset(
    df_name_sample)
test_df = train_model.create_dataset(
    df_name_sample.sample(n=50))


embedding_dim = 5
model_kwargs = {"embedding_dim": embedding_dim}

model_dict = {}
model_list = ['TransE','TransH','TransF','TransR','TransD']
for model_name in model_list:
    experiment_name = model_name+f"_dim{embedding_dim}"
    model = train_model.create_pipeline(train_df, test_df,
                        model_name, model_kwargs, experiment_name)
    model_dict[model_name] = model

In [9]:
def filter_candidates(candidates_df:pd.DataFrame, threshold:float) -> pd.DataFrame:
    candidates_sample_df = candidates_df[candidates_df['distance']<threshold]
    candidates_sample_df = candidates_sample_df[['Head','Relation','Tail']]
    return candidates_sample_df

def compute_missing_df(original_df:pd.DataFrame, sample_df:pd.DataFrame) -> pd.DataFrame:
    df_missing = original_df[~df_name.apply(tuple, axis=1).isin(sample_df.apply(tuple, axis=1))]
    return df_missing

def compute_coverage(filtred_df :pd.DataFrame, df_missing:pd.DataFrame) -> float:
    df_coverage = filtred_df[filtred_df.apply(tuple, axis=1).isin(df_missing.apply(tuple, axis=1))]
    coverage = len(df_coverage) / len(df_missing)
    return coverage

def compute_cand_completness(filtred_df :pd.DataFrame, df_missing:pd.DataFrame) -> float:
    df_coverage = filtred_df[filtred_df.apply(tuple, axis=1).isin(df_missing.apply(tuple, axis=1))]
    coverage = len(df_coverage) / len(df_missing)
    return coverage

In [24]:
transe_score_dict = {}
candidates_df = triple_gen.generate_all_candidates(df_name_sample)
model = model_dict['TransE']
list_dist = get_list_dist(candidates_df, model.model, train_df)
candidates_df['distance'] = list_dist
proportion_dict = {}

mean_dist = candidates_df['distance'].mean()
std_dist = candidates_df['distance'].std()

threshold_list = [mean_dist- std_dist, mean_dist-0.5*std_dist,mean_dist,
                mean_dist+0.5*std_dist,mean_dist + std_dist]
for threshold in threshold_list:
    filtred_df = filter_candidates(candidates_df, threshold)
    df_missing = compute_missing_df(df_name, df_name_sample)
    coverage = compute_coverage(filtred_df, df_missing)
    reduction_ratio = len(filtred_df)/len(candidates_df)
    new_score = coverage/reduction_ratio
    transe_score_dict[threshold] = new_score, reduction_ratio, coverage
    proportion_dict[threshold] = reduction_ratio

In [22]:
for key in transe_score_dict.keys():
    score = format(transe_score_dict[key][0],'.2f')
    reduction_ratio = format(1- transe_score_dict[key][1],'.2f')
    coverage = format(transe_score_dict[key][2],'.2f')
    rr_cov = format((1- transe_score_dict[key][1])*transe_score_dict[key][2],'.2f')
    print(f'Treshold:{format(key,'.2f')}\t score:{score}\t RR:{reduction_ratio}\t Coverage:{coverage}\t RR*Cov:{rr_cov}')

Treshold:1.06	 score:2.34	 RR:0.84	 Coverage:0.38	 RR*Cov:0.32
Treshold:1.49	 score:1.82	 RR:0.65	 Coverage:0.63	 RR*Cov:0.41
Treshold:1.92	 score:1.42	 RR:0.43	 Coverage:0.80	 RR*Cov:0.35
Treshold:2.35	 score:1.27	 RR:0.29	 Coverage:0.89	 RR*Cov:0.26
Treshold:2.78	 score:1.17	 RR:0.19	 Coverage:0.95	 RR*Cov:0.18


In [25]:
def compute_cov_rr(candidates_df, model):
    score_dict = {}
    list_dist = get_list_dist(candidates_df, model.model, train_df)
    candidates_df['distance'] = list_dist

    mean_dist = candidates_df['distance'].mean()
    std_dist = candidates_df['distance'].std()

    threshold_list = [mean_dist- std_dist, mean_dist-0.5*std_dist,mean_dist,
                    mean_dist+0.5*std_dist,mean_dist + std_dist]
    for threshold in threshold_list:
        filtred_df = filter_candidates(candidates_df, threshold)
        df_missing = compute_missing_df(df_name, df_name_sample)
        coverage = compute_coverage(filtred_df, df_missing)
        reduction_ratio = len(filtred_df)/len(candidates_df)
        new_score = coverage/reduction_ratio
        score_dict[threshold] = new_score, reduction_ratio, coverage
    return score_dict

In [26]:
model = model_dict['TransH']
score_dict = compute_cov_rr(candidates_df, model)

In [27]:
for key in score_dict.keys():
    score = format(score_dict[key][0],'.2f')
    reduction_ratio = format(1- score_dict[key][1],'.2f')
    coverage = format(score_dict[key][2],'.2f')
    rr_cov = format((1- score_dict[key][1])*score_dict[key][2],'.2f')
    print(f'Treshold:{format(key,'.2f')}\t score:{score}\t RR:{reduction_ratio}\t Coverage:{coverage}\t RR*Cov:{rr_cov}')

Treshold:0.56	 score:5.61	 RR:0.96	 Coverage:0.20	 RR*Cov:0.19
Treshold:1.48	 score:1.64	 RR:0.59	 Coverage:0.67	 RR*Cov:0.40
Treshold:2.40	 score:1.20	 RR:0.30	 Coverage:0.83	 RR*Cov:0.25
Treshold:3.33	 score:1.13	 RR:0.22	 Coverage:0.88	 RR*Cov:0.19
Treshold:4.25	 score:1.12	 RR:0.17	 Coverage:0.92	 RR*Cov:0.16


In [28]:
model = model_dict['TransD']
trand_score_dict = compute_cov_rr(candidates_df, model)

print('TransD')
for key in score_dict.keys():
    score = format(score_dict[key][0],'.2f')
    reduction_ratio = format(1- score_dict[key][1],'.2f')
    coverage = format(score_dict[key][2],'.2f')
    rr_cov = format((1- score_dict[key][1])*score_dict[key][2],'.2f')
    print(f'Treshold:{format(key,'.2f')}\t score:{score}\t RR:{reduction_ratio}\t Coverage:{coverage}\t RR*Cov:{rr_cov}')

TransD
Treshold:0.56	 score:5.61	 RR:0.96	 Coverage:0.20	 RR*Cov:0.19
Treshold:1.48	 score:1.64	 RR:0.59	 Coverage:0.67	 RR*Cov:0.40
Treshold:2.40	 score:1.20	 RR:0.30	 Coverage:0.83	 RR*Cov:0.25
Treshold:3.33	 score:1.13	 RR:0.22	 Coverage:0.88	 RR*Cov:0.19
Treshold:4.25	 score:1.12	 RR:0.17	 Coverage:0.92	 RR*Cov:0.16
